In [1]:
"""
Prepare working GeoPackage for chain logistics (copy + temporary filling costs).

Behavior:
1) verify source file exists,
2) create/overwrite chain_with_cost.gpkg with only resell and filling,
3) set temporary split filling costs in filling layer (cost_fil_kg_out / cost_fil_kg_in).


time for the run: 7 min
"""

from __future__ import annotations

import sqlite3
from pathlib import Path

import geopandas as gpd
import pandas as pd

DATA_DIR = Path("dataset_big")
SOURCE_GPKG_PATH = DATA_DIR / "full_lpg_chain_nig_3857.gpkg"
OUTPUT_GPKG_PATH = DATA_DIR / "chain_with_cost.gpkg"

RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
FILL_COST_OUT_COL = "cost_fil_kg_out"
FILL_COST_IN_COL = "cost_fil_kg_in"
LEGACY_COST_COL = "cost_kg"
TEMP_FILLING_COST_OUT = 0.6
TEMP_FILLING_COST_IN = 0.5

print("[1/4] Checking source GeoPackage...")
if not SOURCE_GPKG_PATH.exists():
    raise FileNotFoundError(f"Source file not found: {SOURCE_GPKG_PATH}")

print("[2/4] Creating reduced copied GeoPackage (only resell + filling)...")
if OUTPUT_GPKG_PATH.exists():
    OUTPUT_GPKG_PATH.unlink()

with sqlite3.connect(str(SOURCE_GPKG_PATH)) as conn:
    cur = conn.cursor()
    layers = cur.execute(
        "SELECT table_name FROM gpkg_contents WHERE data_type IN ('features', 'attributes')"
    ).fetchall()
    source_layer_names = {name for (name,) in layers}

required_layers = {RESELL_LAYER, FILLING_LAYER}
missing_source_layers = required_layers.difference(source_layer_names)
if missing_source_layers:
    raise KeyError(
        f"Missing required source layer(s): {sorted(missing_source_layers)} in {SOURCE_GPKG_PATH}"
    )

resell_src = gpd.read_file(SOURCE_GPKG_PATH, layer=RESELL_LAYER)
filling_src = gpd.read_file(SOURCE_GPKG_PATH, layer=FILLING_LAYER)
# Remove empty geometries to avoid issues with SQL spatial functions
resell_src = resell_src[~resell_src.geometry.is_empty]
filling_src = filling_src[~filling_src.geometry.is_empty]
if resell_src.empty:
    raise RuntimeError(f"Layer '{RESELL_LAYER}' is empty in source file.")
if filling_src.empty:
    raise RuntimeError(f"Layer '{FILLING_LAYER}' is empty in source file.")

resell_src.to_file(OUTPUT_GPKG_PATH, layer=RESELL_LAYER, driver="GPKG")
filling_src.to_file(OUTPUT_GPKG_PATH, layer=FILLING_LAYER, driver="GPKG")
print(
    f"Created reduced copy: {OUTPUT_GPKG_PATH} | "
    f"resell={len(resell_src):,}, filling={len(filling_src):,}"
)

print("[3/4] Validating layers in reduced copied GeoPackage...")
with sqlite3.connect(str(OUTPUT_GPKG_PATH)) as conn:
    cur = conn.cursor()
    layers = cur.execute(
        "SELECT table_name FROM gpkg_contents WHERE data_type IN ('features', 'attributes')"
    ).fetchall()
    layer_names = {name for (name,) in layers}
    if not layer_names:
        raise RuntimeError(f"No layers found in copied file: {OUTPUT_GPKG_PATH}")
    if FILLING_LAYER not in layer_names:
        raise KeyError(f"Required layer '{FILLING_LAYER}' not found in {OUTPUT_GPKG_PATH}")

print("[4/4] Updating filling-point cost columns in copied file...")
filling_out = gpd.read_file(OUTPUT_GPKG_PATH, layer=FILLING_LAYER)
# Remove empty geometries before updating costs
filling_out = filling_out[~filling_out.geometry.is_empty]
if filling_out.empty:
    raise RuntimeError(f"Layer '{FILLING_LAYER}' is empty in copied file.")

filling_out[FILL_COST_OUT_COL] = float(TEMP_FILLING_COST_OUT)
filling_out[FILL_COST_IN_COL] = float(TEMP_FILLING_COST_IN)
filling_out[FILL_COST_OUT_COL] = pd.to_numeric(filling_out[FILL_COST_OUT_COL], errors="coerce").astype(float)
filling_out[FILL_COST_IN_COL] = pd.to_numeric(filling_out[FILL_COST_IN_COL], errors="coerce").astype(float)

# Keep legacy column for compatibility only.
filling_out[LEGACY_COST_COL] = filling_out[FILL_COST_OUT_COL].astype(float)
filling_out.to_file(OUTPUT_GPKG_PATH, layer=FILLING_LAYER, driver="GPKG")

print(f"Done. Updated {len(filling_out):,} filling rows in copied file.")
print(f"Set {FILL_COST_OUT_COL}={TEMP_FILLING_COST_OUT} and {FILL_COST_IN_COL}={TEMP_FILLING_COST_IN}")
print(f"Source unchanged: {SOURCE_GPKG_PATH}")
print(f"Output file:      {OUTPUT_GPKG_PATH}")

[1/4] Checking source GeoPackage...
[2/4] Creating reduced copied GeoPackage (only resell + filling)...
Created reduced copy: dataset_big\chain_with_cost.gpkg | resell=2,413, filling=375
[3/4] Validating layers in reduced copied GeoPackage...
[4/4] Updating filling-point cost columns in copied file...
Done. Updated 375 filling rows in copied file.
Set cost_fil_kg_out=0.6 and cost_fil_kg_in=0.5
Source unchanged: dataset_big\full_lpg_chain_nig_3857.gpkg
Output file:      dataset_big\chain_with_cost.gpkg


In [2]:
from __future__ import annotations

import heapq
import sqlite3
import time
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from scipy.spatial import cKDTree

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
DATA_DIR = Path("dataset_big")
WORK_GPKG = DATA_DIR / "chain_with_cost.gpkg"
RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
MOTO_FRICTION_RASTER = DATA_DIR / "friction_moto.tif"

ID_COL = "id_res&fil"
OUT_REF_COL = "filling_reference"
OUT_TIME_COL = "filling_ref_time_min"

CELL_SIZE_METERS = 1000.0
USE_8_NEIGHBORS = False
NODATA_INT = -1
UNASSIGNED_TIME_MIN = 7000.0

# ---------------------------------------------------------------------
# Helper: read friction raster
# ---------------------------------------------------------------------
def _read_friction(path: Path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32, copy=False)
        nodata = src.nodata
        profile = src.profile.copy()
    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr).astype(np.float32)
    arr = np.where(arr > 0, arr, np.nan).astype(np.float32)
    return arr, profile


# ---------------------------------------------------------------------
# Helper: map points to raster grid
# ---------------------------------------------------------------------
def _map_points_to_grid(gdf: gpd.GeoDataFrame, transform, width: int, height: int):
    rows, cols = rasterio.transform.rowcol(
        transform, gdf.geometry.x.values, gdf.geometry.y.values
    )
    rows = np.asarray(rows, dtype=np.int32)
    cols = np.asarray(cols, dtype=np.int32)
    inside = (rows >= 0) & (rows < height) & (cols >= 0) & (cols < width)
    return rows, cols, inside


# ---------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------
t0 = time.time()

print("[1/6] Reading friction raster and building valid mask...")
if not WORK_GPKG.exists():
    raise FileNotFoundError(f"Missing working GeoPackage: {WORK_GPKG}")
if not MOTO_FRICTION_RASTER.exists():
    raise FileNotFoundError(f"Missing raster: {MOTO_FRICTION_RASTER}")

friction, ref_profile = _read_friction(MOTO_FRICTION_RASTER)
height, width = friction.shape
transform = ref_profile["transform"]
crs = ref_profile["crs"]

valid = np.isfinite(friction)
if not np.any(valid):
    raise RuntimeError("No valid cells in motorized friction raster.")
print(f"Grid: {width} x {height} | valid cells: {int(valid.sum()):,}")

print("[2/6] Loading reseller and filling points...")
resell = gpd.read_file(WORK_GPKG, layer=RESELL_LAYER)
filling = gpd.read_file(WORK_GPKG, layer=FILLING_LAYER)
if resell.empty:
    raise RuntimeError("Resell layer is empty.")
if filling.empty:
    raise RuntimeError("Filling layer is empty.")

for name, gdf in ((RESELL_LAYER, resell), (FILLING_LAYER, filling)):
    if ID_COL not in gdf.columns:
        raise KeyError(f"Missing '{ID_COL}' in layer '{name}'.")
    if gdf.geometry.isna().all():
        raise RuntimeError(f"All geometries are null in layer '{name}'.")

if resell.crs != crs:
    resell = resell.to_crs(crs)
if filling.crs != crs:
    filling = filling.to_crs(crs)

resell = resell[
    resell.geometry.notna() & resell.geometry.geom_type.isin(["Point"])
].copy()
filling = filling[
    filling.geometry.notna() & filling.geometry.geom_type.isin(["Point"])
].copy()

rid_resell = pd.to_numeric(resell[ID_COL], errors="coerce")
rid_fill = pd.to_numeric(filling[ID_COL], errors="coerce")
if rid_resell.isna().any() or (rid_resell <= 0).any() or (not rid_resell.is_unique):
    raise ValueError(
        f"Layer '{RESELL_LAYER}' must have unique positive numeric '{ID_COL}'."
    )
if rid_fill.isna().any() or (rid_fill <= 0).any() or (not rid_fill.is_unique):
    raise ValueError(
        f"Layer '{FILLING_LAYER}' must have unique positive numeric '{ID_COL}'."
    )

print("[3/6] Mapping points to friction grid...")
res_rows, res_cols, in_res = _map_points_to_grid(resell, transform, width, height)
fill_rows, fill_cols, in_fill = _map_points_to_grid(filling, transform, width, height)

resell = resell.loc[in_res].copy()
filling = filling.loc[in_fill].copy()
res_rows = res_rows[in_res]
res_cols = res_cols[in_res]
fill_rows = fill_rows[in_fill]
fill_cols = fill_cols[in_fill]
rid_resell = rid_resell.loc[in_res].astype(np.int32).to_numpy()
rid_fill = rid_fill.loc[in_fill].astype(np.int32).to_numpy()

if len(resell) == 0 or len(filling) == 0:
    raise RuntimeError("No points left inside raster extent after mapping.")

print(f"Resellers on grid: {len(resell):,}")
print(f"Fillings on grid : {len(filling):,}")

print("[4/6] Building shortest-path graph on friction_moto...")
node_id = -np.ones((height, width), dtype=np.int32)
vr, vc = np.where(valid)
node_id[vr, vc] = np.arange(len(vr), dtype=np.int32)
n_nodes = len(vr)

if USE_8_NEIGHBORS:
    neighbors = [
        (-1, 0), (1, 0), (0, -1), (0, 1),
        (-1, -1), (-1, 1), (1, -1), (1, 1)
    ]
else:
    neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1)]

edge_i = []
edge_j = []
edge_c = []      # costo temporale (minuti)
edge_km = []     # distanza geometrica (km)   <-- NUOVO
diag_factor = np.sqrt(2.0)

for r, c in zip(vr, vc):
    n0 = node_id[r, c]
    f0 = friction[r, c]
    for dr, dc in neighbors:
        rr = r + dr
        cc = c + dc
        if rr < 0 or rr >= height or cc < 0 or cc >= width:
            continue
        n1 = node_id[rr, cc]
        if n1 < 0:
            continue
        f1 = friction[rr, cc]
        if not np.isfinite(f1):
            continue

        step_m = CELL_SIZE_METERS
        if dr != 0 and dc != 0:
            step_m *= diag_factor

        # Tempo di percorrenza (minuti) = friction media * distanza in metri / 60? No, friction è già in minuti per metro? 
        # Nel codice originale: cost = 0.5*(f0+f1)*step_m (step_m in metri, friction in min/m? Devi verificare unità. 
        # Assumo che friction sia in minuti per metro (come da definizione originale). Quindi cost è in minuti.
        cost = 0.5 * (f0 + f1) * step_m
        if cost <= 0:
            continue

        # Distanza geometrica (km)
        km = step_m / 1000.0

        edge_i.append(int(n0))
        edge_j.append(int(n1))
        edge_c.append(float(cost))
        edge_km.append(float(km))

graph_time = csr_matrix((edge_c, (edge_i, edge_j)), shape=(n_nodes, n_nodes))
graph_km   = csr_matrix((edge_km, (edge_i, edge_j)), shape=(n_nodes, n_nodes))

n_comp, comp = connected_components(csgraph=graph_time, directed=False, return_labels=True)
print(f"Graph nodes: {n_nodes:,} | edges: {len(edge_i):,} | components: {n_comp:,}")


# Map resellers and fillings to graph nodes
res_nodes = node_id[res_rows, res_cols]
fill_nodes = node_id[fill_rows, fill_cols]

ok_res = res_nodes >= 0
ok_fill = fill_nodes >= 0
res_nodes = res_nodes[ok_res]
res_rows = res_rows[ok_res]
res_cols = res_cols[ok_res]
rid_resell = rid_resell[ok_res]

fill_nodes = fill_nodes[ok_fill]
fill_rows = fill_rows[ok_fill]
fill_cols = fill_cols[ok_fill]
rid_fill = rid_fill[ok_fill]

if len(res_nodes) == 0 or len(fill_nodes) == 0:
    raise RuntimeError("No reseller/filling mapped to valid graph nodes.")

print("[5/6] Running multi‑source Dijkstra from all filling plants...")

# Multi‑source Dijkstra (exact minimum travel time)
INF = np.inf
dist = np.full(n_nodes, INF, dtype=np.float64)      # tempo (minuti)
dist_km = np.full(n_nodes, INF, dtype=np.float64)   # distanza (km)   <-- NUOVO
source_id = np.full(n_nodes, NODATA_INT, dtype=np.int32)
finalized = np.zeros(n_nodes, dtype=bool)

# Priority queue entries: (distance, node_index, source_filling_id)
heap = []
for fn, fid in zip(fill_nodes, rid_fill):
    fn = int(fn)
    if dist[fn] > 0:
        dist[fn] = 0.0
        dist_km[fn] = 0.0                           # <-- NUOVO
        source_id[fn] = int(fid)
        heapq.heappush(heap, (0.0, fn, int(fid)))

# Process queue
while heap:
    d, u, src = heapq.heappop(heap)
    if finalized[u]:
        continue
    if d > dist[u]:
        continue
    finalized[u] = True

    # Esplora i vicini usando il grafo dei tempi
    for v in graph_time[u].indices:
        if finalized[v]:
            continue
        w_time = graph_time[u, v]
        w_km   = graph_km[u, v]                     # <-- NUOVO
        if not np.isfinite(w_time):
            continue
        new_d = d + w_time
        if new_d < dist[v]:
            dist[v] = new_d
            dist_km[v] = dist_km[u] + w_km          # <-- NUOVO
            source_id[v] = src
            heapq.heappush(heap, (new_d, v, src))

# Assign each reseller the nearest filling plant
n_res = len(res_nodes)
ref_id = np.full(n_res, NODATA_INT, dtype=np.int32)
ref_time = np.full(n_res, UNASSIGNED_TIME_MIN, dtype=np.float32)
ref_dist = np.full(n_res, np.nan, dtype=np.float32)   # <-- NUOVO

for i, node in enumerate(res_nodes):
    node = int(node)
    if dist[node] < UNASSIGNED_TIME_MIN:
        ref_id[i] = source_id[node]
        ref_time[i] = float(dist[node])
        ref_dist[i] = float(dist_km[node])            # <-- NUOVO

assigned = (ref_id >= 0) & (ref_time < UNASSIGNED_TIME_MIN)
print(f"Assigned resellers: {int(assigned.sum()):,}/{n_res:,}")

print("[6/6] Updating reseller table in working GeoPackage...")

# Prepare updates DataFrame with distance
updates = pd.DataFrame({
    ID_COL: rid_resell,
    OUT_REF_COL: ref_id,
    OUT_TIME_COL: ref_time,
    "filling_ref_distance_km": ref_dist,          # <-- NUOVO
})

# Read existing layer
resell_gdf = gpd.read_file(WORK_GPKG, layer=RESELL_LAYER)

# Create columns if missing
if OUT_REF_COL not in resell_gdf.columns:
    resell_gdf[OUT_REF_COL] = NODATA_INT
if OUT_TIME_COL not in resell_gdf.columns:
    resell_gdf[OUT_TIME_COL] = UNASSIGNED_TIME_MIN
if "filling_ref_distance_km" not in resell_gdf.columns:
    resell_gdf["filling_ref_distance_km"] = np.nan   # <-- NUOVO

# Build update dictionary
update_dict = {
    row[ID_COL]: (row[OUT_REF_COL], row[OUT_TIME_COL], row["filling_ref_distance_km"])
    for _, row in updates.iterrows()
}

# Apply updates
for idx, row in resell_gdf.iterrows():
    res_id = row[ID_COL]
    if res_id in update_dict:
        new_ref, new_time, new_dist = update_dict[res_id]
        resell_gdf.at[idx, OUT_REF_COL] = int(new_ref)
        resell_gdf.at[idx, OUT_TIME_COL] = float(new_time)
        resell_gdf.at[idx, "filling_ref_distance_km"] = float(new_dist)   # <-- NUOVO

# Save back
resell_gdf.to_file(WORK_GPKG, layer=RESELL_LAYER, driver="GPKG")

print(f"Updated layer: {WORK_GPKG} | {RESELL_LAYER}")
print(f"- Column added/updated: {OUT_REF_COL}")
print(f"- Column added/updated: {OUT_TIME_COL}")
print(f"- Column added/updated: filling_ref_distance_km")               # <-- NUOVO
print(f"Done in {(time.time() - t0)/60:.1f} min")


[1/6] Reading friction raster and building valid mask...
Grid: 1442 x 1156 | valid cells: 1,078,258
[2/6] Loading reseller and filling points...
[3/6] Mapping points to friction grid...
Resellers on grid: 2,413
Fillings on grid : 375
[4/6] Building shortest-path graph on friction_moto...
Graph nodes: 1,078,258 | edges: 4,302,586 | components: 20
[5/6] Running multi‑source Dijkstra from all filling plants...
Assigned resellers: 2,413/2,413
[6/6] Updating reseller table in working GeoPackage...
Updated layer: dataset_big\chain_with_cost.gpkg | resell
- Column added/updated: filling_reference
- Column added/updated: filling_ref_time_min
- Column added/updated: filling_ref_distance_km
Done in 2.2 min


In [3]:
"""
Quick QA on reseller -> filling assignment (compact) with distance.
"""

from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

DATA_DIR = Path("dataset_big")
WORK_GPKG = DATA_DIR / "chain_with_cost.gpkg"
RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
ID_COL = "id_res&fil"
FILL_REF_COL = "filling_reference"
FILL_TIME_COL = "filling_ref_time_min"
FILL_DIST_COL = "filling_ref_distance_km"   # <-- NUOVO

resell = gpd.read_file(WORK_GPKG, layer=RESELL_LAYER)
filling = gpd.read_file(WORK_GPKG, layer=FILLING_LAYER)

res_ref = pd.to_numeric(resell[FILL_REF_COL], errors="coerce")
res_tmin = pd.to_numeric(resell[FILL_TIME_COL], errors="coerce")
res_dist = pd.to_numeric(resell[FILL_DIST_COL], errors="coerce")   # <-- NUOVO
fill_ids = pd.to_numeric(filling[ID_COL], errors="coerce")
fill_id_set = set(fill_ids[np.isfinite(fill_ids)].astype(np.int64).tolist())

valid_ref = np.isfinite(res_ref) & (res_ref > 0) & pd.Series(res_ref.astype("Int64")).isin(fill_id_set).to_numpy()
valid_time = np.isfinite(res_tmin) & (res_tmin >= 0) & (res_tmin < 7000)
valid_dist = np.isfinite(res_dist) & (res_dist >= 0) & (res_dist < 7000)   # <-- NUOVO
valid = valid_ref & valid_time & valid_dist

n = len(resell)
print("=== QUICK ASSIGNMENT QA ===")
print(f"Valid assignments: {int(valid.sum()):,}/{n:,} ({100.0*valid.mean():.2f}%)")
print(f"Invalid reference: {int((~valid_ref).sum()):,}")
print(f"Invalid time:      {int((~valid_time).sum()):,}")
print(f"Invalid distance:  {int((~valid_dist).sum()):,}")   # <-- NUOVO

if int(valid.sum()) > 0:
    t = res_tmin[valid].to_numpy(dtype=float)
    d = res_dist[valid].to_numpy(dtype=float)               # <-- NUOVO
    print(
        "Time min/p50/p95/p99/max (min): "
        f"{float(np.nanmin(t)):.2f} / {float(np.nanmedian(t)):.2f} / "
        f"{float(np.nanpercentile(t, 95)):.2f} / {float(np.nanpercentile(t, 99)):.2f} / "
        f"{float(np.nanmax(t)):.2f}"
    )
    print(
        "Distance min/p50/p95/p99/max (km): "
        f"{float(np.nanmin(d)):.2f} / {float(np.nanmedian(d)):.2f} / "
        f"{float(np.nanpercentile(d, 95)):.2f} / {float(np.nanpercentile(d, 99)):.2f} / "
        f"{float(np.nanmax(d)):.2f}"
    )

=== QUICK ASSIGNMENT QA ===
Valid assignments: 2,413/2,413 (100.00%)
Invalid reference: 0
Invalid time:      0
Invalid distance:  0
Time min/p50/p95/p99/max (min): 0.00 / 4.00 / 81.84 / 176.63 / 1087.94
Distance min/p50/p95/p99/max (km): 0.00 / 5.00 / 121.40 / 244.88 / 458.00


In [4]:
"""
Build filling_point_clients from 4.3 output + reseller->filling linkage.
"""

from __future__ import annotations

from pathlib import Path

import geopandas as gpd
import pandas as pd

DATA_DIR = Path("dataset_big")
SELL_POINT_GPKG = DATA_DIR / "sell_point_clients.gpkg"
SELL_POINT_LAYER = "sell_point_clients"
CHAIN_GPKG = DATA_DIR / "chain_with_cost.gpkg"
RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
OUTPUT_GPKG = DATA_DIR / "filling_point_clients.gpkg"
OUTPUT_LAYER = "filling_point_clients"

ID_COL = "id_res&fil"
FILL_REF_COL = "filling_reference"

CLIENT_COLS = [
    "clients_walk",
    "clients_max_ideal_walk",
    "clients_car",
    "clients_max_ideal_car",
    "clients",
    "clients_max_ideal",
]

print("[1/7] Loading filling base from chain_with_cost...")
filling_44 = gpd.read_file(CHAIN_GPKG, layer=FILLING_LAYER)
if filling_44.empty:
    raise RuntimeError(f"Layer '{FILLING_LAYER}' is empty in {CHAIN_GPKG}")
if ID_COL not in filling_44.columns:
    raise KeyError(f"Missing '{ID_COL}' in layer '{FILLING_LAYER}' of {CHAIN_GPKG}")

filling_44[ID_COL] = pd.to_numeric(filling_44[ID_COL], errors="coerce")
filling_44 = filling_44[filling_44[ID_COL].notna() & (filling_44[ID_COL] > 0)].copy()
filling_44[ID_COL] = filling_44[ID_COL].astype("int64")
if filling_44.empty:
    raise RuntimeError("No valid filling IDs found in filling layer.")

filling_base_cols = [c for c in [ID_COL, "place_id", "lat", "lon", "geometry"] if c in filling_44.columns]
filling_base = filling_44[filling_base_cols].copy()
filling_id_set = set(filling_base[ID_COL].tolist())
print(f"Filling points in chain_with_cost: {len(filling_base):,}")

print("[2/7] Loading sell_point_clients from 4.3...")
sell_points = gpd.read_file(SELL_POINT_GPKG, layer=SELL_POINT_LAYER)
if sell_points.empty:
    raise RuntimeError(f"Layer '{SELL_POINT_LAYER}' is empty in {SELL_POINT_GPKG}")

required = [ID_COL] + CLIENT_COLS
missing = [c for c in required if c not in sell_points.columns]
if missing:
    raise KeyError(f"Missing required column(s) in sell_point_clients: {missing}")

sell_points[ID_COL] = pd.to_numeric(sell_points[ID_COL], errors="coerce")
sell_points = sell_points[sell_points[ID_COL].notna() & (sell_points[ID_COL] > 0)].copy()
sell_points[ID_COL] = sell_points[ID_COL].astype("int64")
for c in CLIENT_COLS:
    sell_points[c] = pd.to_numeric(sell_points[c], errors="coerce").fillna(0.0).astype(float)

print("[3/7] Attaching local filling metrics from 4.3 by id_res&fil...")
marker_candidates = ["id_filling_only", "id_fillingonly", "id_fillingOnly"]
marker_col = next((c for c in marker_candidates if c in sell_points.columns), None)

local_cols = [ID_COL] + CLIENT_COLS
for c in ["place_id", "lat", "lon"]:
    if c in sell_points.columns and c not in local_cols:
        local_cols.append(c)
if marker_col is not None and marker_col not in local_cols:
    local_cols.append(marker_col)

local_metrics = sell_points[local_cols].copy()
local_metrics = local_metrics.drop_duplicates(subset=[ID_COL], keep="first")

out = filling_base.merge(local_metrics, on=ID_COL, how="left", suffixes=("", "_sp"))
for c in CLIENT_COLS:
    out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0.0).astype(float)
if marker_col is None:
    out["id_filling_only"] = pd.NA
    marker_col = "id_filling_only"

print("[4/7] Aggregating all assigned reseller clients per filling...")
resell_44 = gpd.read_file(CHAIN_GPKG, layer=RESELL_LAYER)
for c in [ID_COL, FILL_REF_COL]:
    if c not in resell_44.columns:
        raise KeyError(f"Missing '{c}' in layer '{RESELL_LAYER}' of {CHAIN_GPKG}")

res_assign = resell_44[[ID_COL, FILL_REF_COL]].copy()
res_assign[ID_COL] = pd.to_numeric(res_assign[ID_COL], errors="coerce")
res_assign[FILL_REF_COL] = pd.to_numeric(res_assign[FILL_REF_COL], errors="coerce")
res_assign = res_assign[
    res_assign[ID_COL].notna()
    & res_assign[FILL_REF_COL].notna()
    & (res_assign[ID_COL] > 0)
    & (res_assign[FILL_REF_COL] > 0)
]
res_assign[ID_COL] = res_assign[ID_COL].astype("int64")
res_assign[FILL_REF_COL] = res_assign[FILL_REF_COL].astype("int64")
res_assign = res_assign[res_assign[FILL_REF_COL].isin(filling_id_set)].copy()

point_clients = sell_points[[ID_COL, "clients", "clients_max_ideal"]].copy()
assigned = res_assign.merge(point_clients, on=ID_COL, how="left")
assigned["clients"] = pd.to_numeric(assigned["clients"], errors="coerce").fillna(0.0).astype(float)
assigned["clients_max_ideal"] = pd.to_numeric(assigned["clients_max_ideal"], errors="coerce").fillna(0.0).astype(float)

agg = (
    assigned.groupby(FILL_REF_COL, as_index=False)[["clients", "clients_max_ideal"]]
    .sum()
    .rename(
        columns={
            FILL_REF_COL: ID_COL,
            "clients": "assigned_fil_clients",
            "clients_max_ideal": "assigned_fil_max_ideal_clients",
        }
    )
)

print("[5/7] Building final filling table...")
out = out.merge(agg, on=ID_COL, how="left")
out["assigned_fil_clients"] = pd.to_numeric(out["assigned_fil_clients"], errors="coerce").fillna(0.0).astype(float)
out["assigned_fil_max_ideal_clients"] = pd.to_numeric(out["assigned_fil_max_ideal_clients"], errors="coerce").fillna(0.0).astype(float)

out["total_fil_clients"] = out["clients"].astype(float) + out["assigned_fil_clients"]
out["total_max_ideal_clients"] = out["clients_max_ideal"].astype(float) + out["assigned_fil_max_ideal_clients"]

keep_cols = [
    ID_COL,
    marker_col,
    "place_id",
    "lat",
    "lon",
    "clients_walk",
    "clients_max_ideal_walk",
    "clients_car",
    "clients_max_ideal_car",
    "clients",
    "clients_max_ideal",
    "total_fil_clients",
    "total_max_ideal_clients",
    "geometry",
]
keep_cols = [c for c in keep_cols if c in out.columns]
out = out[keep_cols].copy()
out = gpd.GeoDataFrame(out, geometry="geometry", crs=filling_44.crs)

print("[6/7] Totals check vs 4.3 output...")
sum_43_clients = float(sell_points["clients"].sum())
sum_43_max = float(sell_points["clients_max_ideal"].sum())
sum_fill_clients = float(out["total_fil_clients"].sum())
sum_fill_max = float(out["total_max_ideal_clients"].sum())

diff_clients = sum_fill_clients - sum_43_clients
diff_max = sum_fill_max - sum_43_max

print("=== TOTALS COMPARISON (filling aggregation vs sell_point_clients) ===")
print(f"4.3 total clients             : {sum_43_clients:,.2f}")
print(f"4.35 total_fil_clients        : {sum_fill_clients:,.2f}")
print(f"Difference                    : {diff_clients:+,.6f}")
print(f"4.3 total clients_max_ideal   : {sum_43_max:,.2f}")
print(f"4.35 total_max_ideal_clients  : {sum_fill_max:,.2f}")
print(f"Difference                    : {diff_max:+,.6f}")

tol = 1e-6
if abs(diff_clients) <= tol and abs(diff_max) <= tol:
    print("Check result: OK (totals match)")
else:
    print(
        "Check result: WARNING (totals differ). "
        "Possible causes: points without valid filling_reference or IDs not aligned across steps."
    )

print("[7/7] Writing filling_point_clients output...")
out.to_file(OUTPUT_GPKG, layer=OUTPUT_LAYER, driver="GPKG")
print(f"Saved: {OUTPUT_GPKG} | layer={OUTPUT_LAYER}")
print(f"Filling rows written: {len(out):,}")

[1/7] Loading filling base from chain_with_cost...
Filling points in chain_with_cost: 375
[2/7] Loading sell_point_clients from 4.3...
[3/7] Attaching local filling metrics from 4.3 by id_res&fil...
[4/7] Aggregating all assigned reseller clients per filling...
[5/7] Building final filling table...
[6/7] Totals check vs 4.3 output...
=== TOTALS COMPARISON (filling aggregation vs sell_point_clients) ===
4.3 total clients             : 26,489,816.81
4.35 total_fil_clients        : 26,489,816.81
Difference                    : -0.000000
4.3 total clients_max_ideal   : 205,374,940.04
4.35 total_max_ideal_clients  : 205,374,940.04
Difference                    : -0.000000
Check result: OK (totals match)
[7/7] Writing filling_point_clients output...
Saved: dataset_big\filling_point_clients.gpkg | layer=filling_point_clients
Filling rows written: 375
